# EDA - Bộ dữ liệu dự đoán tiểu đường

Nguồn: diabetes_prediction_dataset.csv (Kaggle - Mohammed Mustafa). 100.000 dòng, 8 đặc trưng và nhãn diabetes.
Notebook khảo sát dữ liệu và chốt cách tiền xử lý. Giải thích chi tiết xem mo_ta.txt.


In [23]:
import csv, os

CSV_PATH = os.path.join("data", "diabetes_prediction_dataset.csv")

def doc_du_lieu(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def ti_le_mac(records):
    if not records:
        return 0.0
    return 100.0 * sum(1 for r in records if r["diabetes"] == "1") / len(records)

def in_bang(tieu_de, cac_nhom):
    print(tieu_de)
    for ten, nhom in cac_nhom:
        print(f"  {ten:<12}: n={len(nhom):>6}  ty le mac={ti_le_mac(nhom):>6.2f}%")

rows = doc_du_lieu(CSV_PATH)
len(rows)

100000

## 1. Tổng quan

In [24]:
print("So dong:", len(rows))
print("Cot    :", list(rows[0].keys()))
print("O trong:", sum(1 for r in rows for v in r.values() if not v.strip()))

So dong: 100000
Cot    : ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
O trong: 0


## 2. Phân bố nhãn

In [25]:
n = len(rows)
mac = sum(1 for r in rows if r["diabetes"] == "1")
print(f"Khong mac: {n-mac:>6}  ({100.0*(n-mac)/n:.2f}%)")
print(f"Mac      : {mac:>6}  ({100.0*mac/n:.2f}%)")

Khong mac:  91500  (91.50%)
Mac      :   8500  (8.50%)


Dữ liệu mất cân bằng 91.5/8.5. Đánh giá bằng precision, recall, F1 và mốc đoán đa số; không dùng riêng accuracy.


## 3. Chất lượng dữ liệu

In [29]:
cols = list(rows[0].keys())
seen, trung = set(), 0
for r in rows:
    k = tuple(r[c] for c in cols)
    if k in seen:
        trung += 1
    else:
        seen.add(k)
print("Trung lap        :", trung, "-> con", len(rows) - trung)

Trung lap        : 3854 -> con 96146


Xử lý:
Trùng lặp 3.854 dòng: loại, còn 96.146.

## 4. Tỷ lệ mắc theo từng chỉ số

In [27]:
tuoi = lambda r: float(r["age"])
in_bang("Tuoi (nguong ADA/CDC Diabetes Risk Test):", [
    ("<40",   [r for r in rows if tuoi(r) < 40]),
    ("40-49", [r for r in rows if 40 <= tuoi(r) < 50]),
    ("50-59", [r for r in rows if 50 <= tuoi(r) < 60]),
    (">=60",  [r for r in rows if tuoi(r) >= 60]),
])
print()

hba1c = lambda r: float(r["HbA1c_level"])
in_bang("HbA1c (nguong ADA):", [
    ("<5.7",    [r for r in rows if hba1c(r) < 5.7]),
    ("5.7-6.4", [r for r in rows if 5.7 <= hba1c(r) < 6.5]),
    (">=6.5",   [r for r in rows if hba1c(r) >= 6.5]),
])
print()

glu = lambda r: float(r["blood_glucose_level"])
in_bang("Glucose (nguong ADA):", [
    ("<100",    [r for r in rows if glu(r) < 100]),
    ("100-125", [r for r in rows if 100 <= glu(r) < 126]),
    (">=126",   [r for r in rows if glu(r) >= 126]),
])
print()

bmi = lambda r: float(r["bmi"])
in_bang("BMI (nguong WHO):", [
    ("<18.5",     [r for r in rows if bmi(r) < 18.5]),
    ("18.5-24.9", [r for r in rows if 18.5 <= bmi(r) < 25]),
    ("25-29.9",   [r for r in rows if 25 <= bmi(r) < 30]),
    (">=30",      [r for r in rows if bmi(r) >= 30]),
])

Tuoi (nguong ADA/CDC Diabetes Risk Test):
  <40         : n= 45487  ty le mac=  1.56%
  40-49       : n= 14595  ty le mac=  6.84%
  50-59       : n= 14863  ty le mac= 12.51%
  >=60        : n= 25055  ty le mac= 19.68%

HbA1c (nguong ADA):
  <5.7        : n= 37857  ty le mac=  0.00%
  5.7-6.4     : n= 41346  ty le mac=  8.00%
  >=6.5       : n= 20797  ty le mac= 24.96%

Glucose (nguong ADA):
  <100        : n= 21119  ty le mac=  0.00%
  100-125     : n=  7025  ty le mac=  0.00%
  >=126       : n= 71856  ty le mac= 11.83%

BMI (nguong WHO):
  <18.5       : n=  8494  ty le mac=  0.75%
  18.5-24.9   : n= 22219  ty le mac=  3.88%
  25-29.9     : n= 45751  ty le mac=  7.30%
  >=30        : n= 23536  ty le mac= 17.99%


In [28]:
in_bang("Cao huyet ap:", [
    ("khong", [r for r in rows if r["hypertension"] == "0"]),
    ("co",    [r for r in rows if r["hypertension"] == "1"]),
])
print()

in_bang("Benh tim:", [
    ("khong", [r for r in rows if r["heart_disease"] == "0"]),
    ("co",    [r for r in rows if r["heart_disease"] == "1"]),
])
print()

in_bang("Gioi tinh:", [
    (g, [r for r in rows if r["gender"] == g])
    for g in sorted(set(r["gender"] for r in rows))
])
print()

in_bang("Hut thuoc:", [
    (h, [r for r in rows if r["smoking_history"] == h])
    for h in sorted(set(r["smoking_history"] for r in rows))
])

Cao huyet ap:
  khong       : n= 92515  ty le mac=  6.93%
  co          : n=  7485  ty le mac= 27.90%

Benh tim:
  khong       : n= 96058  ty le mac=  7.53%
  co          : n=  3942  ty le mac= 32.14%

Gioi tinh:
  Female      : n= 58552  ty le mac=  7.62%
  Male        : n= 41430  ty le mac=  9.75%
  Other       : n=    18  ty le mac=  0.00%

Hut thuoc:
  No Info     : n= 35816  ty le mac=  4.06%
  current     : n=  9286  ty le mac= 10.21%
  ever        : n=  4004  ty le mac= 11.79%
  former      : n=  9352  ty le mac= 17.00%
  never       : n= 35095  ty le mac=  9.53%
  not current : n=  6447  ty le mac= 10.70%


Tỷ lệ mắc tăng dần theo HbA1c, glucose, tuổi và BMI; cao huyết áp và bệnh tim làm tỷ lệ mắc tăng khoảng 4 lần.
HbA1c < 5.7 và glucose < 126 gần như không có ca mắc, là điều kiện cần nhưng chưa đủ: glucose >= 126 phủ khoảng 72% dữ liệu mà tỷ lệ mắc chỉ 11.8%.
Nhãn phụ thuộc nhiều yếu tố nên chọn logistic regression để tổ hợp các yếu tố thay vì dùng một ngưỡng đơn.


## 5. Tổng hợp tiền xử lý

| Vấn đề | Xử lý | Lý do |
|---|---|---|
| Trùng lặp 3.854 dòng | Loại, còn 96.146 | Không thêm thông tin |
| Mất cân bằng 91.5/8.5 | Đánh giá bằng precision/recall/F1 | Accuracy không phản ánh nhóm bệnh |
| Biến hạng mục (gender, smoking) | One-hot | Không gán thứ hạng số tùy tiện |
| Đặc trưng số | Chuẩn hóa theo train | Ổn định khi huấn luyện logistic |
